# Gated Cross-Modal Learning: Hierarchical Classification with XLSX Labels
## Political Meme Classification (Tamil/Malayalam) with Level 1 & Level 2 Ablation Study

This notebook implements a gated cross-modal model for **hierarchical classification** using labels stored in **XLSX files** with two levels (Level 1: broad category, Level 2: sub-category).

**Key Features:**
- Reads hierarchical labels from xlsx files (Image_id, Image_name, Level 1, Level 2)
- Dual classification heads (one per level)
- Comprehensive ablation study (4 model variants)
- OCR text extraction with caching
- Full reproducibility with train/val/test splits

## 1. Environment Setup & Dependencies

Import all libraries and verify Excel support is available.

In [ ]:
# ============== INSTALL ocr_tamil (Run if getting import errors) ==============
# ocr_tamil is the Tamil OCR library used in the reference notebook
# It works reliably for extracting Tamil text from meme images

def install_ocr_tamil():
    """Install ocr_tamil if not already installed"""
    import subprocess
    import sys
    
    try:
        from ocr_tamil.ocr import OCR
        print("✓ ocr_tamil already installed")
        return True
    except ImportError:
        print("Installing ocr_tamil...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "ocr_tamil"])
            print("✓ ocr_tamil installed successfully")
            return True
        except Exception as e:
            print(f"⚠ Failed to install ocr_tamil: {e}")
            print("  Manual install: pip install ocr_tamil")
            return False

# Uncomment the line below to auto-install ocr_tamil
# install_ocr_tamil()

print("✓ ocr_tamil installation helper ready")
print("  To use: Uncomment install_ocr_tamil() and run this cell")

## 1.5. Install ocr_tamil (Tamil OCR Library)

This notebook uses **`ocr_tamil`** for Tamil text extraction - the same library used in the reference notebook.

**Installation:**
```bash
pip install ocr_tamil
```

Or run cell 1.5 and uncomment `install_ocr_tamil()` to auto-install.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
import warnings
import json
import re
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import defaultdict

warnings.filterwarnings('ignore')

# Excel support for XLSX hierarchical labels
try:
    import openpyxl
    HAS_OPENPYXL = True
    print("✓ openpyxl available for Excel support")
except ImportError:
    HAS_OPENPYXL = False
    print("⚠ WARNING: openpyxl not installed!")
    print("Install with: pip install openpyxl")

# Tamil OCR support (same as reference notebook)
try:
    from ocr_tamil.ocr import OCR
    HAS_OCR_TAMIL = True
    print("✓ ocr_tamil available for Tamil text extraction")
except ImportError:
    HAS_OCR_TAMIL = False
    print("⚠ ocr_tamil not installed. Install with: pip install ocr_tamil")

# Set seeds and device
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

## 2. Configuration: XLSX File Paths & Data Settings

Define paths to your XLSX label files and image directory. The notebook supports two workflows:

**Workflow A: Separate Validation XLSX**
- Have: `TRAIN.xlsx`, `VAL.xlsx`, `TEST.xlsx` (separate files)
- Set: `USE_SPLIT_FROM_TRAIN = False`

**Workflow B: Split Train Internally** (Your case)
- Have: `TRAIN.xlsx`, `TEST.xlsx` (only train and test)
- Set: `USE_SPLIT_FROM_TRAIN = True`
- Notebook will split TRAIN.xlsx into train/val using `VAL_SPLIT = 0.2`

In [ ]:
# ============== PATH CONFIGURATION ==============
USE_REAL_DATA = True  # Set to True when ready to use real XLSX files
USE_HIERARCHICAL = True  # Always True for this notebook (XLSX format)

# Tamil OCR settings (using ocr_tamil library - same as reference notebook)
USE_TAMIL_OCR = True  # Set to True to extract Tamil text from images
OCR_METHOD = 'ocr_tamil'  # Uses ocr_tamil library (reference notebook approach)

# Split strategy
USE_SPLIT_FROM_TRAIN = True  # If True: split from train XLSX; if False: use separate VAL XLSX
VAL_SPLIT = 0.25  # Validation split size when splitting from train
SPLIT_RANDOM_STATE = 42
STRATIFY_BY_LEVEL = 2  # Stratify by 'Level 2' (1 for Level 1, 2 for Level 2)

# XLSX file paths (UPDATE THESE with your actual paths)
TRAIN_EXCEL_PATH = r"/content/dataset/Train/Train_labels.xlsx"  # Must have: Image_id, Image_name, Level 1, Level 2
VAL_EXCEL_PATH = r"/content/dataset/Test/Test_labels.xlsx"      # Only used if USE_SPLIT_FROM_TRAIN = False
TEST_EXCEL_PATH = r"/content/dataset/Test/Test_labels.xlsx"     # Must have same columns as train

# Image directory paths (separate train and test folders)
TRAIN_IMAGE_DIR_PATH = r"/content/dataset/Train/Train_images"   # Directory with train/val image files (jpg/png)
TEST_IMAGE_DIR_PATH = r"/content/dataset/Test/Test_images"      # Directory with test image files (jpg/png)

# Optional OCR caches (CSV files with columns: image_name, ocr_text)
# If provided, OCR extraction will be skipped and cached results will be used
TRAIN_OCR_CACHE = None  # e.g., "ocr_train_cache.csv" to skip re-extraction
VAL_OCR_CACHE = None
TEST_OCR_CACHE = None

# DataLoader settings
BATCH_SIZE = 32
NUM_WORKERS = 0

# Training hyperparameters
NUM_EPOCHS = 15
LEARNING_RATE = 4e-5
NUM_LEVEL1_CLASSES = 2  # E.g., Support, Troll/Oppose
NUM_LEVEL2_CLASSES = 4  # E.g., 5 sub-categories per level 1

print(f"Config: USE_REAL_DATA={USE_REAL_DATA}, USE_HIERARCHICAL={USE_HIERARCHICAL}")
print(f"Config: USE_SPLIT_FROM_TRAIN={USE_SPLIT_FROM_TRAIN}, VAL_SPLIT={VAL_SPLIT}")
print(f"Config: USE_TAMIL_OCR={USE_TAMIL_OCR}, OCR_METHOD={OCR_METHOD}")

## 3. Load & Validate Labels from XLSX Files

Read XLSX files, verify required columns, clean label values, and create label-to-index mappings.

In [ ]:
def load_and_validate_xlsx(excel_path, data_split='train', require_labels=True):
    """
    Load XLSX file and validate required columns.
    
    Args:
        excel_path: Path to XLSX file
        data_split: Name of split ('train', 'val', 'test')
        require_labels: If False, don't drop rows with null labels (for test set)
    
    Expected columns: Image_id (or meme_id), Image_name, Level 1, Level 2
    """
    if not HAS_OPENPYXL:
        raise ImportError("openpyxl required. Install with: pip install openpyxl")
    
    df = pd.read_excel(excel_path)
    print(f"\n✓ Loaded {data_split} XLSX: {len(df)} rows")
    print(f"  Columns: {list(df.columns)}")
    
    # Standardize column names
    col_mapping = {
        'Image_id': 'image_id', 'meme_id': 'image_id',
        'Image_name': 'image_name',
        'Level 1': 'level1', 'Level1': 'level1',
        'Level 2': 'level2', 'Level2': 'level2'
    }
    df = df.rename(columns=col_mapping)
    
    # Validate required columns
    required_cols = ['image_name']
    if require_labels:
        required_cols.extend(['level1', 'level2'])
    
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing Required columns: {missing}. Found: {list(df.columns)}")
    
    # Ensure image_id exists
    if 'image_id' not in df.columns:
        df['image_id'] = range(len(df))
    
    # Clean: remove null labels (only for train/val, not test)
    if require_labels:
        initial_len = len(df)
        df = df.dropna(subset=['image_name', 'level1', 'level2'])
        if len(df) < initial_len:
            print(f"  ⚠ Dropped {initial_len - len(df)} rows with null labels")
        
        # Clean: strip whitespace from label strings
        df['level1'] = df['level1'].astype(str).str.strip()
        df['level2'] = df['level2'].astype(str).str.strip()
        
        print(f"  ✓ After validation: {len(df)} rows")
        print(f"  Level 1 classes: {df['level1'].nunique()} unique values")
        print(f"    {dict(df['level1'].value_counts())}")
        print(f"  Level 2 classes: {df['level2'].nunique()} unique values")
    else:
        # For test set without labels
        print(f"  ✓ Test set (no label validation): {len(df)} rows")
        if 'level1' not in df.columns:
            df['level1'] = 'unknown'
            df['level2'] = 'unknown'
    
    # Always clean image_name
    df['image_name'] = df['image_name'].astype(str).str.strip()
    
    return df

print("✓ XLSX loader function ready")

## 4. Build Label Encoders for Hierarchical Labels

Map Level 1 and Level 2 strings to integer indices for training.

In [ ]:
def build_label_encoders(train_df):
    """
    Create label-to-index mappings from training data.
    Ensures all levels have consistent integer indices.
    """
    # Get unique label values from training split
    level1_classes = sorted(train_df['level1'].unique())
    level2_classes = sorted(train_df['level2'].unique())
    
    # Create mappings
    level1_to_idx = {label: idx for idx, label in enumerate(level1_classes)}
    level2_to_idx = {label: idx for idx, label in enumerate(level2_classes)}
    
    # Inverse mappings for reporting
    idx_to_level1 = {idx: label for label, idx in level1_to_idx.items()}
    idx_to_level2 = {idx: label for label, idx in level2_to_idx.items()}
    
    print(f"\n✓ Label Encodings:")
    print(f"  Level 1 ({len(level1_to_idx)} classes):")
    for label, idx in sorted(level1_to_idx.items()):
        print(f"    {idx}: {label}")
    print(f"  Level 2 ({len(level2_to_idx)} classes):")
    for label, idx in sorted(level2_to_idx.items()):
        print(f"    {idx}: {label}")
    
    return level1_to_idx, level2_to_idx, idx_to_level1, idx_to_level2

print("✓ Label encoder builder ready")

In [ ]:
# ============== TEXT CLEANING UTILITIES ==============
class TextCleaner:
    """Comprehensive text preprocessing and cleaning utility"""
    
    @staticmethod
    def remove_urls(text):
        """Remove URLs from text"""
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        return re.sub(url_pattern, '', text)
    
    @staticmethod
    def remove_emails(text):
        """Remove email addresses"""
        email_pattern = r'[\w\.-]+@[\w\.-]+\.\w+'
        return re.sub(email_pattern, '', text)
    
    @staticmethod
    def remove_special_chars(text, keep_spaces=True):
        """Remove special characters, keep alphanumeric and spaces"""
        if keep_spaces:
            text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?]', '', text)
        else:
            text = re.sub(r'[^a-zA-Z0-9]', '', text)
        return text
    
    @staticmethod
    def remove_extra_whitespace(text):
        """Remove extra whitespaces and normalize"""
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    @staticmethod
    def remove_numbers(text):
        """Remove numeric characters"""
        return re.sub(r'\d+', '', text)
    
    @staticmethod
    def lowercase(text):
        """Convert text to lowercase"""
        return text.lower()
    
    @staticmethod
    def remove_duplicate_chars(text, max_repeat=2):
        """Remove duplicate consecutive characters"""
        pattern = r'(.)\1{' + str(max_repeat) + ',}'
        return re.sub(pattern, r'\1' * max_repeat, text)
    
    @staticmethod
    def clean_text(text, remove_urls=True, remove_emails=True, 
                   remove_special_chars=True, lowercase=True, 
                   remove_numbers=False, remove_duplicates=True):
        """Apply comprehensive text cleaning pipeline"""
        if not isinstance(text, str):
            text = str(text)
        
        if remove_urls:
            text = TextCleaner.remove_urls(text)
        if remove_emails:
            text = TextCleaner.remove_emails(text)
        if remove_special_chars:
            text = TextCleaner.remove_special_chars(text)
        if lowercase:
            text = TextCleaner.lowercase(text)
        if remove_numbers:
            text = TextCleaner.remove_numbers(text)
        if remove_duplicates:
            text = TextCleaner.remove_duplicate_chars(text)
        
        text = TextCleaner.remove_extra_whitespace(text)
        return text


# ============== TAMIL OCR TEXT EXTRACTION (using ocr_tamil) ==============
class ImageTextExtractor:
    """Extract Tamil text from images using ocr_tamil (same as reference notebook)"""
    
    def __init__(self, method='ocr_tamil', clean_text=True):
        """
        Initialize Tamil OCR extractor
        
        Args:
            method: 'ocr_tamil' (library used in reference notebook)
            clean_text: Whether to apply text cleaning
        """
        self.method = method
        self.clean_text = clean_text
        self.text_cleaner = TextCleaner()
        self.model = None
        
        if method == 'ocr_tamil':
            if HAS_OCR_TAMIL:
                try:
                    print("Initializing ocr_tamil OCR model...")
                    self.model = OCR(detect=True)
                    print("✓ ocr_tamil model initialized successfully")
                except Exception as e:
                    print(f"⚠ Failed to initialize ocr_tamil: {e}")
                    self.model = None
            else:
                print("⚠ ocr_tamil not available - install with: pip install ocr_tamil")
        else:
            print(f"⚠ Method '{method}' not supported. Use 'ocr_tamil'")
    
    def extract_text(self, image_path):
        """
        Extract text from a single image using ocr_tamil
        
        Args:
            image_path: Path to image file
            
        Returns:
            Extracted text string (cleaned if clean_text=True)
        """
        try:
            if self.method == 'ocr_tamil' and self.model is not None:
                # ocr_tamil.predict returns [list_of_texts]
                results = self.model.predict(str(image_path))
                extracted_text = " ".join(results[0]) if results else ""
            else:
                return ""
            
            # Apply text cleaning if enabled
            if self.clean_text and extracted_text:
                extracted_text = self.text_cleaner.clean_text(extracted_text)
            
            return extracted_text if extracted_text else ""
        
        except Exception as e:
            print(f"Error extracting text from {image_path}: {e}")
            return ""
    
    def batch_extract(self, image_paths, show_progress=True):
        """
        Extract text from multiple images with progress tracking
        
        Args:
            image_paths: List of image file paths
            show_progress: Whether to show progress bar
            
        Returns:
            Dictionary mapping image paths to extracted text
        """
        results = {}
        iterator = tqdm(image_paths, desc="Extracting Tamil OCR") if show_progress else image_paths
        
        for path in iterator:
            text = self.extract_text(path)
            results[str(path)] = text
        
        return results


def cache_ocr_results(image_dir, output_cache_path, ocr_method='ocr_tamil'):
    """
    Pre-cache OCR results for all images in a directory (using ocr_tamil)
    
    Args:
        image_dir: Directory containing images
        output_cache_path: Path to save cache CSV
        ocr_method: OCR method to use (only 'ocr_tamil' supported)
        
    Returns:
        DataFrame with image_name and ocr_text columns
    """
    image_dir = Path(image_dir)
    image_files = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png'))
    
    print(f"Found {len(image_files)} images in {image_dir}")
    
    ocr_extractor = ImageTextExtractor(method=ocr_method, clean_text=True)
    
    if ocr_extractor.model is None:
        print("⚠ OCR model not initialized. Cannot extract text.")
        return None
    
    results = []
    print(f"Extracting Tamil OCR from {len(image_files)} images...")
    
    for img_path in tqdm(image_files, desc="Processing images"):
        text = ocr_extractor.extract_text(img_path)
        results.append({
            'image_name': img_path.name,
            'ocr_text': text
        })
    
    cache_df = pd.DataFrame(results)
    cache_df.to_csv(output_cache_path, index=False)
    print(f"✓ OCR cache saved to {output_cache_path}")
    print(f"  Total: {len(cache_df)} entries")
    print(f"  Non-empty: {(cache_df['ocr_text'].str.len() > 0).sum()} texts extracted")
    
    return cache_df


print("✓ Tamil OCR text extraction utilities defined (using ocr_tamil)")
print("  - TextCleaner with comprehensive preprocessing")
print("  - ImageTextExtractor with ocr_tamil (same as reference notebook)")
print("  - cache_ocr_results() for batch processing")

## 4.5. Tamil OCR Text Extraction Utilities

Extract and clean Tamil text from meme images using **`ocr_tamil`** (same as reference notebook).

**TextCleaner**: Comprehensive text preprocessing
- URL and email removal
- Special character handling
- Whitespace normalization
- Duplicate character removal

**ImageTextExtractor**: Tamil OCR with ocr_tamil
- Robust Tamil text extraction
- Integrated text cleaning
- Batch processing with progress bars
- Caching support for faster reuse

## 5. Dataset Class for Hierarchical XLSX Labels

Load images, extract OCR text, and return encoded hierarchical labels.

In [ ]:
class HierarchicalXLSXDataset(Dataset):
    """Dataset for hierarchical classification from XLSX labels with Tamil OCR (ocr_tamil)."""
    
    def __init__(self, dataframe, image_dir, level1_encoder, level2_encoder,
                 extract_ocr=False, ocr_cache_path=None, transform=None):
        """
        Args:
            dataframe: DataFrame with image_name, level1, level2, image_id
            image_dir: Path to image directory
            level1_encoder: Dict mapping level1 strings to indices
            level2_encoder: Dict mapping level2 strings to indices
            extract_ocr: Whether to extract text via Tamil OCR (ocr_tamil)
            ocr_cache_path: Optional path to pre-extracted OCR cache (CSV with image_name, ocr_text)
            transform: Image transformation pipeline
        """
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.level1_encoder = level1_encoder
        self.level2_encoder = level2_encoder
        self.extract_ocr = extract_ocr
        self.ocr_cache = {}
        self.text_cleaner = TextCleaner()
        
        # Load OCR cache if available
        if ocr_cache_path and Path(ocr_cache_path).exists():
            cache_df = pd.read_csv(ocr_cache_path)
            if 'image_name' in cache_df.columns and 'ocr_text' in cache_df.columns:
                self.ocr_cache = dict(zip(cache_df['image_name'], cache_df['ocr_text']))
                print(f"✓ Loaded OCR cache: {len(self.ocr_cache)} items")
        
        # Initialize Tamil OCR extractor if needed (not using cache)
        if extract_ocr and not self.ocr_cache:
            self.ocr_extractor = ImageTextExtractor(
                method='ocr_tamil',
                clean_text=True
            )
        else:
            self.ocr_extractor = None
        
        # Image transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
        else:
            self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = str(row['image_name'])
        image_id = int(row.get('image_id', idx))
        level1_str = str(row['level1']).strip()
        level2_str = str(row['level2']).strip()
        
        # Load image
        image_path = self.image_dir / image_name
        image_loaded_successfully = False
        try:
            image = Image.open(image_path).convert('RGB')
            image = self.transform(image)
            image_loaded_successfully = True
        except Exception as e:
            print(f"Failed to load {image_path}")
            image = torch.zeros(3, 224, 224)
        
        # Extract Tamil OCR text using ocr_tamil (only if image loaded successfully)
        text = ""
        if self.extract_ocr and image_loaded_successfully:
            if image_name in self.ocr_cache:
                # Use cached OCR result
                text = str(self.ocr_cache[image_name])
            elif self.ocr_extractor:
                # Extract text on-the-fly using Tamil OCR (ocr_tamil)
                try:
                    text = self.ocr_extractor.extract_text(image_path)
                except Exception as e:
                    print(f"Error extracting text from {image_path}: {e}")
                    text = ""
        
        # Fallback to generic text if no OCR text found
        if not text or not text.strip():
            text = "meme_image"
        
        # Encode labels
        level1_idx = self.level1_encoder.get(level1_str, 0)
        level2_idx = self.level2_encoder.get(level2_str, 0)
        
        return {
            'image': image,
            'text': text,
            'level1': torch.tensor(level1_idx, dtype=torch.long),
            'level2': torch.tensor(level2_idx, dtype=torch.long),
            'image_name': image_name,
            'image_id': image_id
        }

print("✓ HierarchicalXLSXDataset class defined with Tamil OCR support (ocr_tamil)")

## 6. Load Data & Create DataLoaders (XLSX Mode)

Load XLSX files, split if needed, encode labels, and create DataLoaders.

In [ ]:
# ============== DATA LOADING & SPLITTING ==============
if USE_REAL_DATA:
    if not HAS_OPENPYXL:
        raise ImportError("openpyxl not installed. Install with: pip install openpyxl")
    
    # Load training data (with labels)
    train_df = load_and_validate_xlsx(TRAIN_EXCEL_PATH, 'train', require_labels=True)
    # Load test data (may not have labels)
    test_df = load_and_validate_xlsx(TEST_EXCEL_PATH, 'test', require_labels=False)
    
    # Build encoders from training data
    level1_encoder, level2_encoder, idx_to_level1, idx_to_level2 = build_label_encoders(train_df)
    NUM_LEVEL1_CLASSES = len(level1_encoder)
    NUM_LEVEL2_CLASSES = len(level2_encoder)
    
    # Encode labels in all splits
    for df in [train_df, test_df]:
        df['level1_idx'] = df['level1'].map(level1_encoder)
        df['level2_idx'] = df['level2'].map(level2_encoder)
    
    # Split strategy
    if USE_SPLIT_FROM_TRAIN:
        # Stratify on level2
        stratify_col = train_df['level2_idx'] if STRATIFY_BY_LEVEL == 2 else train_df['level1_idx']
        
        train_df_split, val_df_split = train_test_split(
            train_df,
            test_size=VAL_SPLIT,
            random_state=SPLIT_RANDOM_STATE,
            stratify=stratify_col
        )
        print(f"\n✓ Split train into:")
        print(f"  Train: {len(train_df_split)} samples")
        print(f"  Val: {len(val_df_split)} samples")
    else:
        val_df_split = load_and_validate_xlsx(VAL_EXCEL_PATH, 'val')
        val_df_split['level1_idx'] = val_df_split['level1'].map(level1_encoder)
        val_df_split['level2_idx'] = val_df_split['level2'].map(level2_encoder)
        train_df_split = train_df
        print(f"\n✓ Using separate val XLSX: {len(val_df_split)} samples")
    
    # Info about Tamil OCR usage
    if USE_TAMIL_OCR:
        if TRAIN_OCR_CACHE:
            print(f"\n📝 Tamil OCR: Using cached results from {TRAIN_OCR_CACHE}")
        else:
            print(f"\n⚙ Tamil OCR: Extracting text on-the-fly using {OCR_METHOD}")
            print(f"  Library: Same as reference notebook")
            print(f"  Note: First epoch will be slower due to OCR extraction")
    else:
        print(f"\n⚠ Tamil OCR disabled (USE_TAMIL_OCR=False)")
    
    # Create datasets with Tamil OCR (using ocr_tamil)
    # Train and Val use TRAIN_IMAGE_DIR_PATH (train images folder)
    train_dataset = HierarchicalXLSXDataset(
        train_df_split, TRAIN_IMAGE_DIR_PATH, level1_encoder, level2_encoder,
        extract_ocr=USE_TAMIL_OCR, ocr_cache_path=TRAIN_OCR_CACHE
    )
    val_dataset = HierarchicalXLSXDataset(
        val_df_split, TRAIN_IMAGE_DIR_PATH, level1_encoder, level2_encoder,
        extract_ocr=USE_TAMIL_OCR, ocr_cache_path=VAL_OCR_CACHE
    )
    # Test uses TEST_IMAGE_DIR_PATH (test images folder)
    test_dataset = HierarchicalXLSXDataset(
        test_df, TEST_IMAGE_DIR_PATH, level1_encoder, level2_encoder,
        extract_ocr=USE_TAMIL_OCR, ocr_cache_path=TEST_OCR_CACHE
    )
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"\n✓ DataLoaders created")
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")

else:
    print("\n⚠ USE_REAL_DATA = False")
    print("  To use your XLSX files with Tamil OCR (ocr_tamil):")
    print("  1. Update paths in cell #3 (config)")
    print("  2. Set USE_REAL_DATA = True")
    print("  3. Set USE_TAMIL_OCR = True (for Tamil text extraction)")
    print("  4. Optional: Provide OCR cache paths to speed up loading")
    print("  5. Re-run this cell")
    print("  ")
    print("  Requirements:")
    print("  - pip install ocr_tamil")
    print("  - pip install openpyxl")
    
    # Create dummy datasets for testing
    print("\n  Creating synthetic dataset for demo...")
    # Placeholder: create small synthetic data
    train_loader = DataLoader([], batch_size=32)
    val_loader = DataLoader([], batch_size=32)

    test_loader = DataLoader([], batch_size=32)    print("  ✓ Synthetic placeholder ready")

In [ ]:
# ============== PRE-CACHE TAMIL OCR (OPTIONAL) ==============
# Uncomment and run this cell once to create OCR cache files using ocr_tamil

"""
# Example: Cache OCR for all images in your dataset
if USE_REAL_DATA and HAS_OCR_TAMIL:
    print("Pre-caching Tamil OCR results using ocr_tamil...")
    
    # Cache OCR results for your image directory
    cache_df = cache_ocr_results(
        image_dir=IMAGE_DIR_PATH,
        output_cache_path="tamil_ocr_cache_all.csv",
        ocr_method='ocr_tamil'
    )
    
    if cache_df is not None:
        print("\n✓ OCR cache created: tamil_ocr_cache_all.csv")
        print("\nNext steps:")
        print("1. Update config to use cache:")
        print("   TRAIN_OCR_CACHE = 'tamil_ocr_cache_all.csv'")
        print("   VAL_OCR_CACHE = 'tamil_ocr_cache_all.csv'")
        print("   TEST_OCR_CACHE = 'tamil_ocr_cache_all.csv'")
        print("2. Re-run data loading cell")
        print("\nSample results:")
        print(cache_df.head(10))
    else:
        print("⚠ Failed to create OCR cache")
else:
    if not USE_REAL_DATA:
        print("⚠ Set USE_REAL_DATA=True to cache OCR")
    elif not HAS_OCR_TAMIL:
        print("⚠ Install ocr_tamil first: pip install ocr_tamil")
"""

print("✓ OCR caching helper ready (uncomment code above to run)")
print("  Uses ocr_tamil - same as reference notebook")

## 6.5. Optional: Pre-Cache Tamil OCR Results

**Speed up training** by pre-extracting OCR text for all images using `ocr_tamil` and saving to CSV.

Run this cell once to create cache files, then update config paths (TRAIN_OCR_CACHE, etc.) to use them.

**Benefits:**
- Tamil OCR extraction only done once
- Faster dataset loading in subsequent runs
- Can reuse across multiple experiments
- Same approach as reference notebook

## 7. Model Architectures for Hierarchical Classification

Define 4 model variants with dual classification heads (Level 1 + Level 2).

In [ ]:
# ============== IMAGE FEATURE EXTRACTOR ==============
class ResNet50FeatureExtractor(nn.Module):
    """ResNet-50 based image feature extractor using ocr_tamil reference approach"""
    def __init__(self, feature_dim=2048, pretrained=True):
        super().__init__()
        resnet = models.resnet50(pretrained=pretrained)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = feature_dim

    def forward(self, x):
        """
        Args:
            x: (B, 3, 224, 224)
        Returns:
            features: (B, 2048)
        """
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return x


# ============== TEXT FEATURE EXTRACTOR ==============
class TextFeatureExtractor(nn.Module):
    """Transformer-based text feature extractor (MuRIL for Tamil support)"""
    def __init__(self, model_name='google/muril-base-cased', feature_dim=768, freeze=True):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.feature_dim = feature_dim
        
        # Freeze transformer params if requested
        if freeze:
            for param in self.model.parameters():
                param.requires_grad = False

    def forward(self, texts):
        """
        Args:
            texts: List of text strings
        Returns:
            text_features: (B, 768) - [CLS] token representation
        """
        # Handle list or single string input
        if isinstance(texts, str):
            texts = [texts]
        elif not isinstance(texts, list):
            texts = list(texts)
        
        try:
            # Tokenize and encode
            encoded = self.tokenizer(texts, return_tensors='pt', padding=True, 
                                    truncation=True, max_length=64).to(self.model.device)
            with torch.no_grad():
                outputs = self.model(**encoded)
            # Use [CLS] token (first token) for document representation
            return outputs.last_hidden_state[:, 0, :]  # [B, 768]
        except Exception as e:
            print(f"Error extracting text features: {e}")
            device = next(self.model.parameters()).device
            return torch.zeros(len(texts), self.feature_dim).to(device)


# ============== CROSS-MODAL ATTENTION ==============
class CrossModalAttention(nn.Module):
    """Bidirectional cross-modal attention between image and text features"""
    def __init__(self, image_dim=2048, text_dim=768, hidden_dim=512):
        super().__init__()
        self.image_dim = image_dim
        self.text_dim = text_dim
        self.hidden_dim = hidden_dim
        
        # Image -> Text attention
        self.image_to_text_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, batch_first=True
        )
        # Text -> Image attention
        self.text_to_image_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=8, batch_first=True
        )
        
        # Projections to hidden dimension
        self.image_proj = nn.Linear(image_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, image_features, text_features):
        """
        Args:
            image_features: (B, 2048)
            text_features: (B, 768)
        Returns:
            attended_image: (B, hidden_dim)
            attended_text: (B, hidden_dim)
        """
        # Project to common dimension
        img_proj = self.image_proj(image_features).unsqueeze(1)  # (B, 1, hidden_dim)
        txt_proj = self.text_proj(text_features).unsqueeze(1)    # (B, 1, hidden_dim)
        
        # Cross-attention: Image queries, Text keys/values
        img_attended, _ = self.image_to_text_attn(img_proj, txt_proj, txt_proj)
        img_attended = self.norm1(img_attended + img_proj)
        
        # Cross-attention: Text queries, Image keys/values
        txt_attended, _ = self.text_to_image_attn(txt_proj, img_proj, img_proj)
        txt_attended = self.norm2(txt_attended + txt_proj)
        
        # Remove sequence dimension
        return img_attended.squeeze(1), txt_attended.squeeze(1)


# ============== GATED FUSION ==============
class GatedFusion(nn.Module):
    """Gated fusion mechanism to combine image and text modalities"""
    def __init__(self, image_dim=512, text_dim=512, fusion_dim=512):
        super().__init__()
        self.image_dim = image_dim
        self.text_dim = text_dim
        self.fusion_dim = fusion_dim
        
        # Gate networks for each modality
        self.image_gate = nn.Sequential(
            nn.Linear(image_dim + text_dim, fusion_dim),
            nn.ReLU(),
            nn.Linear(fusion_dim, 1),
            nn.Sigmoid()
        )
        
        self.text_gate = nn.Sequential(
            nn.Linear(image_dim + text_dim, fusion_dim),
            nn.ReLU(),
            nn.Linear(fusion_dim, 1),
            nn.Sigmoid()
        )
        
        # Fusion projection
        self.fusion_proj = nn.Linear(image_dim + text_dim, fusion_dim)

    def forward(self, image_features, text_features):
        """
        Args:
            image_features: (B, image_dim)
            text_features: (B, text_dim)
        Returns:
            fused: (B, fusion_dim)
            gates: dict with 'image_gate' and 'text_gate'
        """
        # Concatenate features
        combined = torch.cat([image_features, text_features], dim=1)
        
        # Compute gates
        image_gate = self.image_gate(combined)
        text_gate = self.text_gate(combined)
        
        # Apply gates to features
        gated_image = image_features * image_gate
        gated_text = text_features * text_gate
        
        # Fuse gated features
        gated_combined = torch.cat([gated_image, gated_text], dim=1)
        fused = self.fusion_proj(gated_combined)
        
        return fused, {'image_gate': image_gate, 'text_gate': text_gate}


# ============== GATED CROSS-MODAL MODEL (MAIN) ==============
class GatedCrossModalModel(nn.Module):
    """Full gated cross-modal model with hierarchical classification"""
    def __init__(self, num_level1_classes=2, num_level2_classes=5,
                 image_dim=2048, text_dim=768, fusion_dim=512):
        super().__init__()
        self.image_dim = image_dim
        self.text_dim = text_dim
        self.fusion_dim = fusion_dim
        
        # Feature extractors
        self.image_extractor = ResNet50FeatureExtractor(feature_dim=image_dim)
        self.text_extractor = TextFeatureExtractor(feature_dim=text_dim)
        
        # Cross-modal attention
        self.cross_attention = CrossModalAttention(
            image_dim=image_dim,
            text_dim=text_dim,
            hidden_dim=fusion_dim
        )
        
        # Gated fusion
        self.fusion = GatedFusion(
            image_dim=fusion_dim,
            text_dim=fusion_dim,
            fusion_dim=fusion_dim
        )
        
        # Hierarchical classification heads
        self.head_level1 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level1_classes)
        )
        
        self.head_level2 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level2_classes)
        )
        
        self.model_name = "GatedCrossModal (Full Model)"

    def forward(self, images, texts):
        """
        Args:
            images: (B, 3, 224, 224)
            texts: List of text strings
        Returns:
            dict with keys: logits_level1, logits_level2, gates, features
        """
        # Extract features
        img_features = self.image_extractor(images)
        txt_features = self.text_extractor(texts)
        
        # Cross-modal attention
        img_att, txt_att = self.cross_attention(img_features, txt_features)
        
        # Gated fusion
        fused, gates = self.fusion(img_att, txt_att)
        
        # Hierarchical classification
        logits_level1 = self.head_level1(fused)
        logits_level2 = self.head_level2(fused)
        
        return {
            'logits_level1': logits_level1,
            'logits_level2': logits_level2,
            'gates': gates,
            'features': fused
        }


# ============== ABLATION MODELS ==============

class ImageOnlyModel(nn.Module):
    """Baseline: Image-only model without text or cross-modal components"""
    def __init__(self, num_level1_classes=2, num_level2_classes=5, 
                 image_dim=2048, fusion_dim=512):
        super().__init__()
        self.image_extractor = ResNet50FeatureExtractor(feature_dim=image_dim)
        
        self.proj = nn.Sequential(
            nn.Linear(image_dim, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU()
        )
        
        self.head_level1 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level1_classes)
        )
        
        self.head_level2 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level2_classes)
        )
        
        self.model_name = "ImageOnly (Baseline)"

    def forward(self, images, texts=None):
        img_features = self.image_extractor(images)
        features = self.proj(img_features)
        
        logits_level1 = self.head_level1(features)
        logits_level2 = self.head_level2(features)
        
        return {
            'logits_level1': logits_level1,
            'logits_level2': logits_level2,
            'features': features
        }


class AttentionOnlyModel(nn.Module):
    """Ablation: Cross-modal attention without gated fusion"""
    def __init__(self, num_level1_classes=2, num_level2_classes=5,
                 image_dim=2048, text_dim=768, fusion_dim=512):
        super().__init__()
        self.image_extractor = ResNet50FeatureExtractor(feature_dim=image_dim)
        self.text_extractor = TextFeatureExtractor(feature_dim=text_dim)
        
        self.cross_attention = CrossModalAttention(
            image_dim=image_dim,
            text_dim=text_dim,
            hidden_dim=fusion_dim
        )
        
        self.fusion_proj = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU()
        )
        
        self.head_level1 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level1_classes)
        )
        
        self.head_level2 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level2_classes)
        )
        
        self.model_name = "Attention Only (No Gates)"

    def forward(self, images, texts):
        img_features = self.image_extractor(images)
        txt_features = self.text_extractor(texts)
        
        img_att, txt_att = self.cross_attention(img_features, txt_features)
        
        fused_input = torch.cat([img_att, txt_att], dim=1)
        features = self.fusion_proj(fused_input)
        
        logits_level1 = self.head_level1(features)
        logits_level2 = self.head_level2(features)
        
        return {
            'logits_level1': logits_level1,
            'logits_level2': logits_level2,
            'features': features
        }


class GatedFusionOnlyModel(nn.Module):
    """Ablation: Gated fusion without cross-modal attention"""
    def __init__(self, num_level1_classes=2, num_level2_classes=5,
                 image_dim=2048, text_dim=768, fusion_dim=512):
        super().__init__()
        self.image_extractor = ResNet50FeatureExtractor(feature_dim=image_dim)
        self.text_extractor = TextFeatureExtractor(feature_dim=text_dim)
        
        # Project raw features to fusion dimension
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU()
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_dim),
            nn.BatchNorm1d(fusion_dim),
            nn.ReLU()
        )
        
        self.fusion = GatedFusion(
            image_dim=fusion_dim,
            text_dim=fusion_dim,
            fusion_dim=fusion_dim
        )
        
        self.head_level1 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level1_classes)
        )
        
        self.head_level2 = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_level2_classes)
        )
        
        self.model_name = "GatedFusion Only (No Attention)"

    def forward(self, images, texts):
        img_features = self.image_extractor(images)
        txt_features = self.text_extractor(texts)
        
        img_proj = self.image_proj(img_features)
        txt_proj = self.text_proj(txt_features)
        
        fused, gates = self.fusion(img_proj, txt_proj)
        
        logits_level1 = self.head_level1(fused)
        logits_level2 = self.head_level2(fused)
        
        return {
            'logits_level1': logits_level1,
            'logits_level2': logits_level2,
            'gates': gates,
            'features': fused
        }


print("✓ All model architectures defined (4 variants with dual hierarchical heads)")
print("  - GatedCrossModalModel (full model)")
print("  - ImageOnlyModel (baseline)")
print("  - AttentionOnlyModel (no gates)")
print("  - GatedFusionOnlyModel (no attention)")

## 8. Training & Evaluation

Train all model variants and compute metrics on both hierarchical levels.

In [ ]:
# ============== HIERARCHICAL TRAINING ==============
def train_epoch_hierarchical(model, train_loader, optimizer, device, 
                            level1_weight=1.0, level2_weight=1.0):
    """
    Train one epoch with dual hierarchical losses (Level 1 + Level 2)
    
    Args:
        model: PyTorch model with level1/level2 heads
        train_loader: DataLoader with batches
        optimizer: Optimization algorithm
        device: GPU/CPU device
        level1_weight: Loss weight for Level 1
        level2_weight: Loss weight for Level 2
    
    Returns:
        dict with epoch metrics
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    
    total_loss = 0.0
    total_l1_loss = 0.0
    total_l2_loss = 0.0
    l1_correct = 0
    l2_correct = 0
    total_samples = 0
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    
    for batch in pbar:
        # Move data to device
        images = batch['image'].to(device)
        texts = batch['text']
        labels_l1 = batch['level1'].to(device)
        labels_l2 = batch['level2'].to(device)
        
        # Convert texts to list if needed
        if isinstance(texts, torch.Tensor):
            texts = [t for t in texts]
        
        # Forward pass
        outputs = model(images, texts)
        logits_l1 = outputs['logits_level1']
        logits_l2 = outputs['logits_level2']
        
        # Compute losses
        loss_l1 = criterion(logits_l1, labels_l1)
        loss_l2 = criterion(logits_l2, labels_l2)
        loss = level1_weight * loss_l1 + level2_weight * loss_l2
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        total_l1_loss += loss_l1.item()
        total_l2_loss += loss_l2.item()
        
        # Accuracy computation
        l1_preds = torch.argmax(logits_l1, dim=1)
        l2_preds = torch.argmax(logits_l2, dim=1)
        l1_correct += (l1_preds == labels_l1).sum().item()
        l2_correct += (l2_preds == labels_l2).sum().item()
        total_samples += labels_l1.size(0)
        
        pbar.set_postfix({
            'loss': total_loss / (pbar.n + 1),
            'l1_acc': l1_correct / total_samples,
            'l2_acc': l2_correct / total_samples
        })
    
    return {
        'loss': total_loss / len(train_loader),
        'l1_loss': total_l1_loss / len(train_loader),
        'l2_loss': total_l2_loss / len(train_loader),
        'l1_accuracy': l1_correct / total_samples,
        'l2_accuracy': l2_correct / total_samples
    }


# ============== HIERARCHICAL EVALUATION ==============
def evaluate_hierarchical(model, eval_loader, device, idx_to_level1=None, idx_to_level2=None):
    """
    Evaluate model on hierarchical classification with detailed metrics
    
    Args:
        model: Trained PyTorch model
        eval_loader: DataLoader with evaluation batches
        device: GPU/CPU device
        idx_to_level1: Dict mapping level1 indices to class names
        idx_to_level2: Dict mapping level2 indices to class names
    
    Returns:
        dict with comprehensive metrics for both levels
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    all_l1_preds = []
    all_l1_labels = []
    all_l2_preds = []
    all_l2_labels = []
    total_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(eval_loader, desc="Evaluating", leave=False)
        
        for batch in pbar:
            images = batch['image'].to(device)
            texts = batch['text']
            labels_l1 = batch['level1'].to(device)
            labels_l2 = batch['level2'].to(device)
            
            if isinstance(texts, torch.Tensor):
                texts = [t for t in texts]
            
            outputs = model(images, texts)
            logits_l1 = outputs['logits_level1']
            logits_l2 = outputs['logits_level2']
            
            # Loss
            loss = criterion(logits_l1, labels_l1) + criterion(logits_l2, labels_l2)
            total_loss += loss.item()
            
            # Predictions
            l1_preds = torch.argmax(logits_l1, dim=1).cpu().numpy()
            l2_preds = torch.argmax(logits_l2, dim=1).cpu().numpy()
            
            all_l1_preds.extend(l1_preds)
            all_l1_labels.extend(labels_l1.cpu().numpy())
            all_l2_preds.extend(l2_preds)
            all_l2_labels.extend(labels_l2.cpu().numpy())
    
    # Compute metrics for Level 1
    l1_accuracy = accuracy_score(all_l1_labels, all_l1_preds)
    l1_precision, l1_recall, l1_f1, _ = precision_recall_fscore_support(
        all_l1_labels, all_l1_preds, average='weighted', zero_division=0
    )
    l1_cm = confusion_matrix(all_l1_labels, all_l1_preds)
    
    # Compute metrics for Level 2
    l2_accuracy = accuracy_score(all_l2_labels, all_l2_preds)
    l2_precision, l2_recall, l2_f1, _ = precision_recall_fscore_support(
        all_l2_labels, all_l2_preds, average='weighted', zero_division=0
    )
    l2_cm = confusion_matrix(all_l2_labels, all_l2_preds)
    
    return {
        'loss': total_loss / len(eval_loader),
        'level1': {
            'accuracy': l1_accuracy,
            'precision': l1_precision,
            'recall': l1_recall,
            'f1': l1_f1,
            'confusion_matrix': l1_cm,
            'predictions': np.array(all_l1_preds),
            'labels': np.array(all_l1_labels)
        },
        'level2': {
            'accuracy': l2_accuracy,
            'precision': l2_precision,
            'recall': l2_recall,
            'f1': l2_f1,
            'confusion_matrix': l2_cm,
            'predictions': np.array(all_l2_preds),
            'labels': np.array(all_l2_labels)
        }
    }


# ============== ABLATION STUDY RUNNER ==============
def run_ablation_study(models_dict, train_loader, val_loader, test_loader,
                       device, num_epochs=10, learning_rate=1e-4):
    """
    Run full ablation study comparing all model variants
    
    Args:
        models_dict: Dict of {model_name: model_class}
        train_loader, val_loader, test_loader: DataLoaders
        device: GPU/CPU
        num_epochs: Training epochs
        learning_rate: Learning rate
    
    Returns:
        dict with results for all models
    """
    results = {}
    
    for model_name, ModelClass in models_dict.items():
        print(f"\n{'='*60}")
        print(f"Training: {model_name}")
        print(f"{'='*60}")
        
        # Initialize model
        model = ModelClass(
            num_level1_classes=NUM_LEVEL1_CLASSES,
            num_level2_classes=NUM_LEVEL2_CLASSES
        ).to(device)
        
        # Optimizer
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        
        # Training history
        history = {
            'train_loss': [],
            'train_l1_acc': [],
            'train_l2_acc': [],
            'val_loss': [],
            'val_l1_acc': [],
            'val_l2_acc': [],
            'best_val_f1': 0.0
        }
        
        best_model_state = None
        
        # Training loop
        for epoch in range(num_epochs):
            # Train
            train_metrics = train_epoch_hierarchical(
                model, train_loader, optimizer, device
            )
            
            # Validate
            val_metrics = evaluate_hierarchical(model, val_loader, device)
            
            val_f1 = (val_metrics['level1']['f1'] + val_metrics['level2']['f1']) / 2
            scheduler.step(val_f1)
            
            # Record
            history['train_loss'].append(train_metrics['loss'])
            history['train_l1_acc'].append(train_metrics['l1_accuracy'])
            history['train_l2_acc'].append(train_metrics['l2_accuracy'])
            history['val_loss'].append(val_metrics['loss'])
            history['val_l1_acc'].append(val_metrics['level1']['accuracy'])
            history['val_l2_acc'].append(val_metrics['level2']['accuracy'])
            
            # Save best model
            if val_f1 > history['best_val_f1']:
                history['best_val_f1'] = val_f1
                best_model_state = model.state_dict().copy()
            
            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"  Train: L1_Acc={train_metrics['l1_accuracy']:.4f}, L2_Acc={train_metrics['l2_accuracy']:.4f}")
            print(f"  Val:   L1_Acc={val_metrics['level1']['accuracy']:.4f}, L2_Acc={val_metrics['level2']['accuracy']:.4f}, F1={val_f1:.4f}")
        
        # Load best model and evaluate on validation set (final report)
        if best_model_state:
            model.load_state_dict(best_model_state)
        
        final_metrics = evaluate_hierarchical(model, val_loader, device)
        
        results[model_name] = {
            'model': model,
            'history': history,
            'final_metrics': final_metrics
        }
        
        print(f"\nFinal Results on Validation Set ({model_name}):")
        print(f"  Level 1: Acc={final_metrics['level1']['accuracy']:.4f}, F1={final_metrics['level1']['f1']:.4f}")
        print(f"  Level 2: Acc={final_metrics['level2']['accuracy']:.4f}, F1={final_metrics['level2']['f1']:.4f}")
    
    return results


print("✓ Training and evaluation functions ready")
print("  - train_epoch_hierarchical(): Dual loss training")
print("  - evaluate_hierarchical(): Metrics for both levels")
print("  - run_ablation_study(): Full ablation runner")

## 9. Results Analysis & Export

Compare ablation results and export to disk.

In [ ]:
# ============== RESULTS VISUALIZATION ==============
def plot_training_curves(results, output_path='training_curves.png'):
    """Plot training curves for all models"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Training Curves - Hierarchical Classification', fontsize=16, fontweight='bold')
    
    for model_name, data in results.items():
        history = data['history']
        epochs = range(1, len(history['train_loss']) + 1)
        
        axes[0, 0].plot(epochs, history['train_loss'], label=model_name, marker='o')
        axes[0, 1].plot(epochs, history['train_l1_acc'], label=model_name, marker='o')
        axes[1, 0].plot(epochs, history['train_l2_acc'], label=model_name, marker='o')
        axes[1, 1].plot(epochs, history['val_loss'], label=model_name, marker='o')
    
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_title('Level 1 Training Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_title('Level 2 Training Accuracy')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_title('Validation Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved training curves to {output_path}")
    plt.show()


def plot_performance_comparison(results, output_path='performance_comparison.png'):
    """Compare final validation performance across all model variants"""
    model_names = list(results.keys())
    
    l1_accs = [results[m]['final_metrics']['level1']['accuracy'] for m in model_names]
    l1_f1s = [results[m]['final_metrics']['level1']['f1'] for m in model_names]
    l2_accs = [results[m]['final_metrics']['level2']['accuracy'] for m in model_names]
    l2_f1s = [results[m]['final_metrics']['level2']['f1'] for m in model_names]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    x = np.arange(len(model_names))
    width = 0.35
    
    # Level 1
    axes[0].bar(x - width/2, l1_accs, width, label='Accuracy', alpha=0.8)
    axes[0].bar(x + width/2, l1_f1s, width, label='F1 Score', alpha=0.8)
    axes[0].set_xlabel('Model')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Level 1 Performance')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(model_names, rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim([0, 1])
    
    # Level 2
    axes[1].bar(x - width/2, l2_accs, width, label='Accuracy', alpha=0.8)
    axes[1].bar(x + width/2, l2_f1s, width, label='F1 Score', alpha=0.8)
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Score')
    axes[1].set_title('Level 2 Performance')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(model_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved performance comparison to {output_path}")
    plt.show()


def plot_confusion_matrices(results, output_path='confusion_matrices.png'):
    """Plot confusion matrices for all models (Level 2 focus)"""
    model_names = list(results.keys())
    n_models = len(model_names)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, model_name in enumerate(model_names):
        cm = results[model_name]['final_metrics']['level2']['confusion_matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
        axes[idx].set_title(f'{model_name} - Level 2')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('True')
    
    plt.suptitle('Confusion Matrices (Level 2)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved confusion matrices to {output_path}")
    plt.show()


def generate_ablation_report(results, idx_to_level1=None, idx_to_level2=None, 
                            output_path='ablation_report.txt'):
    """Generate comprehensive ablation study report"""
    with open(output_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("ABLATION STUDY: HIERARCHICAL GATED CROSS-MODAL LEARNING\n")
        f.write("="*70 + "\n\n")
        
        # Component analysis
        f.write("ABLATION COMPONENTS:\n")
        f.write("-" * 70 + "\n")
        f.write("1. ImageOnlyModel:      Image features only (baseline)\n")
        f.write("2. AttentionOnlyModel:  Image + Text + Cross-Modal Attention (No Gating)\n")
        f.write("3. GatedFusionOnly:     Image + Text + Gated Fusion (No Attention)\n")
        f.write("4. GatedCrossModal:     Full Model (Attention + Gating)\n\n")
        
        # Results table
        f.write("VALIDATION SET PERFORMANCE (FINAL REPORT):\n")
        f.write("-" * 70 + "\n")
        f.write(f"{'Model':<25} {'L1 Acc':<12} {'L1 F1':<12} {'L2 Acc':<12} {'L2 F1':<12}\n")
        f.write("-" * 70 + "\n")
        
        for model_name in results.keys():
            metrics = results[model_name]['final_metrics']
            l1_acc = metrics['level1']['accuracy']
            l1_f1 = metrics['level1']['f1']
            l2_acc = metrics['level2']['accuracy']
            l2_f1 = metrics['level2']['f1']
            
            f.write(f"{model_name:<25} {l1_acc:<12.4f} {l1_f1:<12.4f} {l2_acc:<12.4f} {l2_f1:<12.4f}\n")
        
        f.write("\n")
        
        # Ablation insights
        f.write("ABLATION ANALYSIS:\n")
        f.write("-" * 70 + "\n")
        
        # Compare ImageOnly vs Full
        img_only = results['ImageOnlyModel']['final_metrics']
        full = results['GatedCrossModalModel']['final_metrics']
        
        l1_gain = (full['level1']['f1'] - img_only['level1']['f1']) / (img_only['level1']['f1'] + 1e-6) * 100
        l2_gain = (full['level2']['f1'] - img_only['level2']['f1']) / (img_only['level2']['f1'] + 1e-6) * 100
        
        f.write(f"\nFull Model vs ImageOnly (Baseline):\n")
        f.write(f"  Level 1 F1 improvement: {l1_gain:+.2f}%\n")
        f.write(f"  Level 2 F1 improvement: {l2_gain:+.2f}%\n")
        
        # Component contributions
        attn = results['AttentionOnlyModel']['final_metrics']
        gates = results['GatedFusionOnlyModel']['final_metrics']
        
        attn_l1_f1 = attn['level1']['f1']
        gates_l1_f1 = gates['level1']['f1']
        full_l1_f1 = full['level1']['f1']
        
        attn_l2_f1 = attn['level2']['f1']
        gates_l2_f1 = gates['level2']['f1']
        full_l2_f1 = full['level2']['f1']
        
        f.write(f"\nComponent Contribution (Level 1 F1):\n")
        f.write(f"  Cross-Modal Attention:  {attn_l1_f1:.4f}\n")
        f.write(f"  Gated Fusion Only:      {gates_l1_f1:.4f}\n")
        f.write(f"  Full Model (Att+Gates): {full_l1_f1:.4f}\n")
        f.write(f"  Synergy:                {full_l1_f1 - max(attn_l1_f1, gates_l1_f1):+.4f}\n")
        
        f.write(f"\nComponent Contribution (Level 2 F1):\n")
        f.write(f"  Cross-Modal Attention:  {attn_l2_f1:.4f}\n")
        f.write(f"  Gated Fusion Only:      {gates_l2_f1:.4f}\n")
        f.write(f"  Full Model (Att+Gates): {full_l2_f1:.4f}\n")
        f.write(f"  Synergy:                {full_l2_f1 - max(attn_l2_f1, gates_l2_f1):+.4f}\n")
        
        f.write("\n" + "="*70 + "\n")
        f.write("RECOMMENDATIONS:\n")
        f.write("="*70 + "\n")
        
        if full_l1_f1 > max(attn_l1_f1, gates_l1_f1):
            f.write("✓ Full gated cross-modal model shows synergistic gains\n")
            f.write("  Both attention and gating mechanisms contribute to improved performance\n")
        elif attn_l1_f1 > gates_l1_f1:
            f.write("✓ Cross-modal attention is the dominant component\n")
            f.write("  Consider increasing attention mechanism capacity\n")
        else:
            f.write("✓ Gated fusion is the dominant component\n")
            f.write("  Consider focusing on gate design and modality weighting\n")
    
    print(f"✓ Saved ablation report to {output_path}")


print("✓ Results visualization and reporting functions ready")
print("  - plot_training_curves()")
print("  - plot_performance_comparison()")
print("  - plot_confusion_matrices()")
print("  - generate_ablation_report()")

## 10. Run Complete Ablation Study

Execute the full ablation study comparing all 4 model variants on your hierarchical dataset.

In [ ]:
# ============== EXECUTE ABLATION STUDY ==============
if USE_REAL_DATA and 'train_loader' in locals():
    print("""
    ╔════════════════════════════════════════════════════════════════╗
    ║  ABLATION STUDY: GATED CROSS-MODAL HIERARCHICAL CLASSIFICATION ║
    ║                                                                ║
    ║  4 Model Variants:                                             ║
    ║  1. ImageOnlyModel        - Baseline (image only)              ║
    ║  2. AttentionOnlyModel    - Cross-modal attention (no gates)   ║
    ║  3. GatedFusionOnlyModel  - Gated fusion (no attention)        ║
    ║  4. GatedCrossModalModel  - Full model (attention + gates)     ║
    ║                                                                ║
    ║  Each model has dual classification heads (Level 1 + Level 2)  ║
    ╚════════════════════════════════════════════════════════════════╝
    """)
    
    # Define all model variants
    models_for_ablation = {
        'ImageOnlyModel': ImageOnlyModel,
        'AttentionOnlyModel': AttentionOnlyModel,
        'GatedFusionOnlyModel': GatedFusionOnlyModel,
        'GatedCrossModalModel': GatedCrossModalModel
    }
    
    # Run ablation study
    print(f"\nStarting ablation study...")
    print(f"  Device: {device}")
    print(f"  Epochs: {NUM_EPOCHS}")
    print(f"  Learning Rate: {LEARNING_RATE}")
    print(f"  Level 1 Classes: {NUM_LEVEL1_CLASSES}")
    print(f"  Level 2 Classes: {NUM_LEVEL2_CLASSES}\n")
    
    ablation_results = run_ablation_study(
        models_for_ablation,
        train_loader,
        val_loader,
        test_loader,
        device,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE
    )
    
    # Generate visualizations
    print("\n" + "="*70)
    print("GENERATING RESULTS & VISUALIZATIONS")
    print("="*70 + "\n")
    
    plot_training_curves(ablation_results)
    plot_performance_comparison(ablation_results)
    plot_confusion_matrices(ablation_results)
    generate_ablation_report(ablation_results, idx_to_level1, idx_to_level2)
    
    print("\n" + "="*70)
    print("ABLATION STUDY COMPLETE")
    print("="*70)
    print("\nOutput files:")
    print("  ✓ training_curves.png")
    print("  ✓ performance_comparison.png")
    print("  ✓ confusion_matrices.png")
    print("  ✓ ablation_report.txt")
    
else:
    print("\n⚠ Cannot run ablation study:")
    if not USE_REAL_DATA:
        print("  - USE_REAL_DATA = False")
        print("  - Set USE_REAL_DATA = True in cell #3 (Configuration)")
        print("  - Ensure all XLSX paths are valid")
    if 'train_loader' not in locals():
        print("  - DataLoaders not initialized")
        print("  - Run cell #7 (Data Loading) first")